# 제출용 파일 1/3 · 5멤버(트리·선형·NN) 학습 — 케글 GPU

해커톤 제출 검증용. 이 파일은 **`train.csv`에서 5개 멤버를 학습**하고 OOF·test 예측을 저장합니다.
(TabM은 파일 2에서 별도 학습, 최종 6멤버 블렌드·`submission_final.csv`는 파일 3에서.)

**생성 멤버:** `lgb(v2v3) · cat(v2v3) · xgb(v3) · lin(ratio) · nn(v2v3)`
(※ `nn`은 트리와 동일한 v2 게이팅·v3 파생을 적용한 버전을 멤버로 저장 — `members_oof["nn"]=oof_nn_v2v3`)

**규정 준수 (test 누수 차단) — 코드로 검증 가능**
- test는 **어떤 학습/전처리 fit에도 미투입.** 명목형 인코딩·스케일·결측 대치·타깃인코딩은 전부 **fold 내부(train fold)에서만 fit** 후 valid/test에 transform.
- 외부 데이터·유사라벨링 미사용. 단일 결정적 실행.

**재현성**
- 트리·선형 = **완전 결정적**(seed 고정 → 동일 환경 bit-동일). NN = 결정성 풀세트 적용(`use_deterministic_algorithms`·`cudnn.deterministic`·`CUBLAS_WORKSPACE_CONFIG`), 단 GPU 종류/드라이버 차이로 소수점 4째자리 흔들림 가능 → **파일 3의 랭크 블렌딩이 흡수**.

**입력:** `train.csv` · `test.csv`   **환경:** ★ GPU 세션(NN). 트리·선형 CPU.
**산출:** `oof_{lgb,cat,xgb,lin,nn}.csv` · `test_{...}.csv` (파일 3 입력)

In [20]:
# ============================================================
# 재현성 고정 (시드 · 결정성 · 버전 스냅샷)
# ============================================================
import os, random
import numpy as np
SEED = 42
os.environ["PYTHONHASHSEED"]="42"; os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
random.seed(SEED); np.random.seed(SEED)
NUM_THREADS = 4   # 트리 결정성 전제. 재실행 시 동일 유지.
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True)   # 엄격 결정성(임베딩 backward까지 결정적 경로 강제). 동일 환경서 NN bit-재현.
except Exception: pass
def snapshot_env(path="env_versions.json"):
    import json,sys,sklearn,scipy,pandas as pd
    v={"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,
       "scikit-learn":sklearn.__version__,"scipy":scipy.__version__}
    for n in ["lightgbm","xgboost","catboost","torch"]:
        try: v[n]=__import__(n).__version__
        except Exception: v[n]=None
    try: json.dump(v,open(path,"w"),indent=2)
    except Exception: pass
    print("[env]",v); return v
ENV=snapshot_env()

[env] {'python': '3.12.13', 'numpy': '2.4.6', 'pandas': '2.3.3', 'scikit-learn': '1.6.1', 'scipy': '1.16.3', 'lightgbm': '4.6.0', 'xgboost': '3.2.0', 'catboost': '1.2.10', 'torch': '2.10.0+cu128'}


## 1. 데이터 로드 + 누수안전 피처 빌더 (행단위·train-only)

In [21]:
import glob, re, json
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier

def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]
    assert h, f"{n} 없음 — Add Input 확인"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
print("train",train.shape,"| test",test.shape,"| base_rate=%.4f"%y.mean())

def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]

# ── 트리 베이스 (명목형 카테고리 코드는 train 카테고리로만 fit) ──
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]  # pandas3.0: str dtype 대비(is_numeric_dtype)
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF)

# ── v2 게이팅 파생 (행단위) ──
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_v2_gated(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"])
    F["g본인_난자수율"]=P6.where(Mk["본인난자"]); F["g기증_난자수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)

# ── 선형용 비율 파생 (행단위) ──
def build_lin_ratios(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아perICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"]); F["g본인_수율"]=P6.where(Mk["본인난자"]); F["g기증_수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr=build_lin_ratios(train); RATte=build_lin_ratios(test)

# ── v3 신규 파생 (행단위) ──
def build_new_derived(df):
    F={}
    tx=NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    return pd.DataFrame(F,index=df.index)
Dtr=build_new_derived(train); Dte=build_new_derived(test)

# ── v3 트리 후보 컬럼 (최종 선택 셋) ──
#   C1=배아이식 유형(EL_set_type), C2=무이식 플래그(FA_*).  C3(ICSI 경로효율)은 혼합난자⊇미세주입난자로 분모 오정의 → 영구 폐기.
V3_TREE_COLS = ["EL_set_type","FA_no_transfer","FA_nontransfer_reason"]
assert all(c in Dtr.columns for c in V3_TREE_COLS)
# 행단위 독립성(누수 0) 검증
assert np.array_equal(build_new_derived(train.head(300)).fillna(-9).values, Dtr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("피처 빌더 준비 ✅ | tf_tree",Xb.shape,"| v2게이팅",V2tr.shape[1],"| 비율",RATtr.shape[1],"| v3트리",len(V3_TREE_COLS))

train (256351, 69) | test (90067, 68) | base_rate=0.2583
피처 빌더 준비 ✅ | tf_tree (256351, 72) | v2게이팅 11 | 비율 15 | v3트리 3


## 2. 트리·선형 멤버 학습기 (fold-내부 fit · 결정적)

In [22]:
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,
         feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50,
         deterministic=True,force_row_wise=True,num_threads=NUM_THREADS)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,
         max_depth=6,subsample=0.8,colsample_bytree=0.8,nthread=NUM_THREADS)
TREE_ITERS=1500
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),TREE_ITERS,
        valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),TREE_ITERS,
        evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=TREE_ITERS,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,
        early_stopping_rounds=80,thread_count=NUM_THREADS); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
def tree_member(kind, extra_tr, extra_te, seed=42):
    """단일 트리 모델 OOF + test (extra_* = 추가 파생 행렬). fold-내부 fit."""
    X  =pd.concat([Xb.reset_index(drop=True),    extra_tr.reset_index(drop=True)],axis=1)
    Xte=pd.concat([Xb_te.reset_index(drop=True), extra_te.reset_index(drop=True)],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]
    o=np.zeros(N); tt=np.zeros(len(Xte))
    for tri,vai in folds:
        m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed); o[vai]=pred_one(kind,m,X.iloc[vai])
        tt+=pred_one(kind,m,Xte)/len(folds)
    return o,tt

# 선형 (타깃인코딩·스케일 전부 fold-내부 fit)
TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)),pr
def lin_member(use_ratios=True,seed=42):
    oof=np.zeros(N); tst=np.zeros(len(test))
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy()
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        if use_ratios:
            Xt=pd.concat([Xt.reset_index(drop=True),RATtr.iloc[tri].reset_index(drop=True)],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True),RATtr.iloc[vai].reset_index(drop=True)],axis=1)
            Xte=pd.concat([Xte.reset_index(drop=True),RATte.reset_index(drop=True)],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return oof,tst
print("트리·선형 학습기 준비 ✅")

트리·선형 학습기 준비 ✅


## 3. NN 멤버 (임베딩-MLP · GPU · fold-내부 fit)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
if DEVICE=="cpu": print("⚠️ GPU 미감지 — Settings>Accelerator>GPU 켜고 재실행 권장")
ORD_CNT=OCC
def prep_global(df):
    df=df.copy()
    for c in ORD_CNT:
        if c in df: df[c]=df[c].map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].map(m)
    return df
def nn_cols(tr,keep):
    keep=[c for c in keep if c in tr.columns and c not in (TARGET,ID_COL)]
    cat=[c for c in keep if not pd.api.types.is_numeric_dtype(tr[c])]; num=[c for c in keep if c not in cat]
    return cat,num
def fit_fold(trdf,cat,num):
    enc={c:{v:i+1 for i,v in enumerate(trdf[c].astype(str).fillna("NA").value_counts().index)} for c in cat}
    med={c:float(trdf[c].median()) for c in num}; mu={}; sd={}
    for c in num:
        x=trdf[c].fillna(med[c]).astype(float); mu[c]=float(x.mean()); sd[c]=float(x.std())+1e-6
    return enc,med,mu,sd
def transform(df,cat,num,enc,med,mu,sd):
    Xc=np.zeros((len(df),len(cat)),dtype=np.int64)
    for j,c in enumerate(cat): Xc[:,j]=df[c].astype(str).fillna("NA").map(enc[c]).fillna(0).astype(int).values
    Xn=np.zeros((len(df),len(num)*2),dtype=np.float32)
    for j,c in enumerate(num):
        Xn[:,2*j+1]=df[c].isna().astype(np.float32).values
        x=df[c].fillna(med[c]).astype(float).values; Xn[:,2*j]=((x-mu[c])/sd[c]).astype(np.float32)
    return Xc,Xn
class EmbMLP(nn.Module):
    def __init__(self,cat_dims,n_num,hidden=(256,128),p=0.2):
        super().__init__()
        self.embs=nn.ModuleList([nn.Embedding(d,min(50,(d+1)//2+1)) for d in cat_dims])
        din=sum(e.embedding_dim for e in self.embs)+n_num; layers=[]
        for h in hidden: layers+=[nn.Linear(din,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(p)]; din=h
        layers+=[nn.Linear(din,1)]; self.mlp=nn.Sequential(*layers)
    def forward(self,xc,xn):
        x=torch.cat([torch.cat([emb(xc[:,i]) for i,emb in enumerate(self.embs)],dim=1),xn],dim=1) if len(self.embs)>0 else xn
        return self.mlp(x).squeeze(1)
def train_nn(Xc_t,Xn_t,yt,Xc_v,Xn_v,yv,cat_dims,epochs=40,patience=6,bs=2048,lr=1e-3,hidden=(256,128),dropout=0.2,seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE=="cuda": torch.cuda.manual_seed_all(seed)
    model=EmbMLP(cat_dims,Xn_t.shape[1],hidden,dropout).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5); lossf=nn.BCEWithLogitsLoss()
    ds=TensorDataset(torch.as_tensor(Xc_t),torch.as_tensor(Xn_t),torch.as_tensor(yt,dtype=torch.float32))
    _g=torch.Generator(); _g.manual_seed(seed)
    def _sw(wid): ws=(torch.initial_seed()+wid)%(2**32); np.random.seed(ws); random.seed(ws)
    dl=DataLoader(ds,batch_size=bs,shuffle=True,drop_last=True,num_workers=0,generator=_g,worker_init_fn=_sw)
    Xcv=torch.as_tensor(Xc_v).to(DEVICE); Xnv=torch.as_tensor(Xn_v).to(DEVICE)
    best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train()
        for xc,xn,yb in dl:
            xc,xn,yb=xc.to(DEVICE),xn.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); loss=lossf(model(xc,xn),yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): pv=torch.sigmoid(model(Xcv,Xnv)).cpu().numpy()
        auc=roc_auc_score(yv,pv)
        if auc>best+1e-5: best=auc; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    model.load_state_dict(best_state); return model,best
def predict_nn(model,Xc,Xn,bs=16384):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(Xc),bs):
            out.append(torch.sigmoid(model(torch.as_tensor(Xc[i:i+bs]).to(DEVICE),torch.as_tensor(Xn[i:i+bs]).to(DEVICE))).cpu().numpy())
    return np.concatenate(out)

# keep 셋 — 순열 중요도(LGBM 홀드아웃, seed 고정 → 결정적)로 nn_base 입력 위생 산출
trg=prep_global(train); teg=prep_global(test)
Xp=train.drop(columns=[TARGET,ID_COL]).copy()
for c in Xp.columns:
    if not pd.api.types.is_numeric_dtype(Xp[c]): Xp[c]=pd.factorize(Xp[c].astype(str))[0]
Xp=Xp.apply(pd.to_numeric,errors="coerce"); fp=list(Xp.columns)
from sklearn.model_selection import train_test_split
a,b,ya,yb=train_test_split(Xp,y,test_size=0.25,stratify=y,random_state=SEED)
mp=lgb.train(dict(objective="binary",metric="auc",learning_rate=0.05,num_leaves=63,verbose=-1,seed=SEED,
                  deterministic=True,force_row_wise=True,num_threads=NUM_THREADS),
             lgb.Dataset(a,ya),num_boost_round=500,valid_sets=[lgb.Dataset(b,yb)],
             callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(0)])
bs0=roc_auc_score(yb,mp.predict(b)); rp=np.random.default_rng(0); dr={}
for c in fp:
    Xq=b.copy(); Xq[c]=Xq[c].values[rp.permutation(len(Xq))]; dr[c]=bs0-roc_auc_score(yb,mp.predict(Xq))
KEEP=[c for c in fp if dr[c]>=0.0001]
CAT,NUM_=nn_cols(trg,KEEP)
print(f"NN keep={len(KEEP)}개 | cat={len(CAT)}(임베딩) num={len(NUM_)}(스케일+결측플래그)")

# NN base OOF/test (동일 fold seed42)
folds=list(StratifiedKFold(5,shuffle=True,random_state=SEED).split(trg,y))
oof_nn=np.zeros(N); test_nn=np.zeros(len(teg))
for k,(tri,vai) in enumerate(folds):
    enc,med,mu,sd=fit_fold(trg.iloc[tri],CAT,NUM_)
    Xc_t,Xn_t=transform(trg.iloc[tri],CAT,NUM_,enc,med,mu,sd)
    Xc_v,Xn_v=transform(trg.iloc[vai],CAT,NUM_,enc,med,mu,sd)
    Xc_e,Xn_e=transform(teg,CAT,NUM_,enc,med,mu,sd)
    dims=[len(enc[c])+1 for c in CAT]
    model,va=train_nn(Xc_t,Xn_t,y[tri],Xc_v,Xn_v,y[vai],dims,seed=SEED)
    oof_nn[vai]=predict_nn(model,Xc_v,Xn_v); test_nn+=predict_nn(model,Xc_e,Xn_e)/5
    print(f"  nn fold{k}: val AUC={va:.5f}")
print(f"nn(base) OOF AUC = {roc_auc_score(y,oof_nn):.5f}")

# ============================================================
# ★ STAGE 추가(Gemini 2순위): nn(v2v3) — 트리와 동일한 v2게이팅+v3 파생변수를
# NN 입력에도 추가해서, NN이 트리와 같은 "무기"를 쓰는 버전을 만듭니다.
# 기존 nn(base)는 그대로 유지(비교용), nn(v2v3)을 별도 멤버로 추가 생성합니다.
# ============================================================
trg_v2v3 = pd.concat([trg.reset_index(drop=True), V2tr.reset_index(drop=True),
                       Dtr[V3_TREE_COLS].reset_index(drop=True)], axis=1)
teg_v2v3 = pd.concat([teg.reset_index(drop=True), V2te.reset_index(drop=True),
                       Dte[V3_TREE_COLS].reset_index(drop=True)], axis=1)

# v2v3 추가 컬럼 중 수치형은 NUM_v2v3에, 범주형(EL_set_type 등은 사실 정수코드라 수치형 취급)
new_cols_v2v3 = list(V2tr.columns) + V3_TREE_COLS
CAT_v2v3, NUM_v2v3_extra = nn_cols(trg_v2v3, KEEP + new_cols_v2v3)
print(f"nn(v2v3) keep={len(KEEP)+len(new_cols_v2v3)}개 | cat={len(CAT_v2v3)} num={len(NUM_v2v3_extra)}")

folds_v2v3 = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(trg_v2v3, y))
oof_nn_v2v3 = np.zeros(N); test_nn_v2v3 = np.zeros(len(teg_v2v3))
for k, (tri, vai) in enumerate(folds_v2v3):
    enc, med, mu, sd = fit_fold(trg_v2v3.iloc[tri], CAT_v2v3, NUM_v2v3_extra)
    Xc_t, Xn_t = transform(trg_v2v3.iloc[tri], CAT_v2v3, NUM_v2v3_extra, enc, med, mu, sd)
    Xc_v, Xn_v = transform(trg_v2v3.iloc[vai], CAT_v2v3, NUM_v2v3_extra, enc, med, mu, sd)
    Xc_e, Xn_e = transform(teg_v2v3, CAT_v2v3, NUM_v2v3_extra, enc, med, mu, sd)
    dims = [len(enc[c]) + 1 for c in CAT_v2v3]
    model, va = train_nn(Xc_t, Xn_t, y[tri], Xc_v, Xn_v, y[vai], dims, seed=SEED)
    oof_nn_v2v3[vai] = predict_nn(model, Xc_v, Xn_v)
    test_nn_v2v3 += predict_nn(model, Xc_e, Xn_e) / 5
    print(f"  nn(v2v3) fold{k}: val AUC={va:.5f}")
print(f"nn(v2v3) OOF AUC = {roc_auc_score(y, oof_nn_v2v3):.5f}  (참고: nn(base)={roc_auc_score(y, oof_nn):.5f})")

NN keep=34개 | cat=6(임베딩) num=28(스케일+결측플래그)
  nn fold0: val AUC=0.73508
  nn fold1: val AUC=0.74081
  nn fold2: val AUC=0.73852


## 4. 5개 멤버 생성 (seed42 · 동일 fold)

In [ ]:
members_oof = {}
members_test = {}

# 1. 트리 모델용 공통 파생변수 결합 (v2 게이팅 + v3 파생변수)
ev2v3_tr = pd.concat([V2tr.reset_index(drop=True), Dtr[V3_TREE_COLS].reset_index(drop=True)], axis=1)
ev2v3_te = pd.concat([V2te.reset_index(drop=True), Dte[V3_TREE_COLS].reset_index(drop=True)], axis=1)

# [1, 2순위] 트리 3종 모델 학습 (LGB, CAT, XGB 모두 동일하게 v2+v3 반영)
print("--- 트리 모델 3종(LGB, CAT, XGB) 학습 시작 ---")
for kind in ["lgb", "cat", "xgb"]:
    o, t = tree_member(kind, ev2v3_tr, ev2v3_te, seed=42)
    members_oof[kind] = o
    members_test[kind] = t
    print(f"  {kind}(v2v3) OOF={roc_auc_score(y, o):.5f}")

# [3순위] LIN 모델 학습
print("--- 3순위: LIN 학습 시작 ---")
o, t = lin_member(use_ratios=True, seed=42)
members_oof["lin"] = o
members_test["lin"] = t
print(f"  lin(ratio) OOF={roc_auc_score(y, o):.5f}")

# [4순위] NN 모델 적용 (앞서 상단 셀에서 생성된 v2v3 결과물 연결)
print("--- 4순위: NN 적용 ---")
members_oof["nn"] = oof_nn_v2v3
members_test["nn"] = test_nn_v2v3
print(f"  nn(v2v3) OOF={roc_auc_score(y, oof_nn_v2v3):.5f}  (참고: nn(base)={roc_auc_score(y, oof_nn):.5f})")

# nn(base)도 별도 파일로 저장 (비교용)
pd.DataFrame({"oof_nn_base": oof_nn, "y": y}).to_csv("oof_nn_base_compare.csv", index=False)

# [최종] 결과 파일 저장
for m in members_oof:
    pd.DataFrame({f"oof_{m}": members_oof[m], "y": y}).to_csv(f"oof_{m}.csv", index=False)
    pd.DataFrame({"ID": test[ID_COL].values, f"test_{m}": members_test[m]}).to_csv(f"test_{m}.csv", index=False)

print("멤버 5종 생성 완료 ✅ (oof_*/test_* 저장)")

## 5. (확인용) 5멤버 랭크블렌드 OOF — 제출 아님

5멤버가 정상인지 점검만. 최종 6멤버 블렌드·`submission_final.csv`는 **파일 3**에서 생성됩니다.

In [ ]:
def _hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get)
    ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm},best[1]
_R={m:runr(members_oof[m]) for m in members_oof}
_w,_b=_hill(_R,y)
print("5멤버 sanity 블렌드 OOF =", round(_b,5), "(nn 포함, ~0.7407 부근이면 정상)")
print("  멤버 OOF:", {m:round(roc_auc_score(y,members_oof[m]),5) for m in members_oof})
print("→ TabM은 파일2, 최종 6멤버 블렌드·submission은 파일3에서.")